# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Metodología para el análisis de interpretabilidad

### 📝 Descripción del notebook
En este notebook se presenta la metodología y mecanismos empleados para realizar el análisis de interpretabilidad de modelos transparantes (Regresión logística y árbol de decisión) y modelos caja negra (SVM, Random Forest y XGBoost). El análisis se divide un una fase de interpretabilidad global, realizada en *06_Global_Interpretability_analysis.ipynb*, y una fase de interpretabilidad local, realizada en *07_Local_Interpretabiliy_analysis.ipynb*.

### Indice de contenidos
1. [Metodología para el análisis de interpretabilidad](#metodologia)
    * [1.1 Interpretabilidad global](#global)
    * [1.2 Interpretabilidad local](#local)

2. [Mecanismos de interpretabilidad](#mecanismos)
    * [2.1 Regresión logística: análisis de coeficientes](#coeficientes)
    * [2.2 Arbol de decisión: reglas de decisión y reducción de impureza](#arbol)
    * [2.3 Modelos de caja negra: SHAP (SHapley Additive exPlanations)](#shap)
        * [2.3.1 Fundamento teórico](#shap_teoria)
        * [2.3.2 Interpretabilidad de los valores SHAP](#shap_interp)

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero jupyter notebook, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [5]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import warnings
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from src.balanced_xgb import BalancedXGBClassifier

# Configuration object
from src.config import cfg

# Production training method
from src.production_training import train_final_model

**Import troubleshooting**

If the `src` imports fail when running this notebook in a different environment,
uncomment and execute the following cell:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [7]:
# Global configuration
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [8]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

<a id="metodologia"></a>
# 1. Metodología para el análisis de interpretabilidad

El objetivo fundamental del análisis de interpretabilidad consiste en auditar cómo los modelos utilizan la información subyacente en los datos para generar sus predicciones. En concreto, se pretende determinar si las predicciones generadas por los modelos se alinean con el conocimiento experto del dominio o si, por el contrario, están influenciadas por patrones complejos cuya interpretación resulta menos evidente desde una perspectiva experta.

Para ello, se emplean los mecanismos de interpretabilidad intrínsecos propios de los modelos transparentes, mientras que en los modelos "caja negra" se utiliza la técnica de interpretabilidad post-hoc SHAP (SHapley Additive exPlanations). De este modo, el análisis de interpretabilidad se situa dentro de la discusión sobre el equilibrio entre capacidad predictiva y explicativa, permitiendo comparar las ventajas y limitaciones de ambos tipos de modelos.

El análisis de interpretabilidad se estructura en dos fases: interpretabilidad global e interpretabilidad local.

<a id="global"></a>
## 1.1 Interpretabilidad global

En primer lugar se realiza un análisis de interpretabilidad global cuyo objetivo es identificar, de manera general, las variables más influyentes en cada modelo y, dada la naturaleza multiclase del problema, aquellas características más relevantes para cada grado de cerramiento. Esta primera fase permite analizar qué características tienen un mayor impacto en el proceso de decisión de los algoritmos y caracterizar conceptualmente cada grado de cerramiento a partir de las variables que más contribuyen a su clasificación.

Este análisis se aplica sobre todos los modelos considerados en el experimento, con el fin de obtener una visión global y comparativa de su comportamiento. De este modo, se pretende identificar si los modelos basan sus predicciones en variables consistentes o si existen diferencias significativas derivadas de la capacidad de cada algoritmo para explotar determinados patrones presentes en los datos. Esto permite analizar si las posibles mejoras de rendimiento de unos modelos frente a otros se basan en el aprovechamiento de información diferente o, por el contrario, de una modelización más compleja de las mismas variables.

En la fase de evaluación de rendimiento se ha empleado una estrategia de *Nested Cross Validation*  con el objetivo de estimar de forma realista la capacidad de generalización de los distintos algoritmos, haciendo uso de todos los datos disponibles. En este esquema, para cada partición externa se obtiene un modelo, configurado con los hiperparámetros optimizados a partir de su conjunto de entrenamiento correspondiente.

Para realizar el análisis de interpretabilidad es necesario disponer de un único modelo final que refleje fielmente el comportamiento aprendidoa partir de los datos. Por este motivo, los modelos utilizados para estudiar la interpretabilidad se entrenan sobre el conjunto de datos completo, con el fin de maximizar la información utilizada para capturar la estructura subyacente del problema. De esta forma, se pretende explicar el comportamiento del modelo que sería desplegado en un entorno de producción, el cual se entrena utilizando todos los datos disponible.

Una alternativa para la selección de hiperparámetros del modelo final consistiría en utilizar la moda de los hiperparámetros obtenidos en las distintas particiones de la *Nested Cross Validation*. Sin embargo, este enfoque heurístico no garantiza la optimalidad global ni asegura que dicha configuración sea la más adecuada para el conjunto de datos completo.

Por este motivo, la configuración final de hiperparámetros se obtiene mediante un proceso de optimización que explora el mismo espacio de búsqueda definido en el experimento de evaluación del rendimiento. Para ello, se emplea una validación cruzada estándar (K-Fold) sobre el conjunto completo de datos.

Con el objetivo de garantizar la consistencia metodológica y la reproducibilidad en la construcción de los modelos finales, esta validación cruzada reutiliza las mismas particiones utilizadas en el bucle externo (Outer Loop) de la Nested Cross Validation. Una vez obtenida la configuración óptima, el algoritmo se reentrena utilizando el conjunto de datos completo.

A continuación se entrenan todos los modelos mediante la funcion *train_final_model()* ubicada en el módulo *src/production_training.py*. Los modelos se almacenan en el directorio *output/models* en formato .pkl.

In [9]:
################################
# Multinomial Logistic Regression
################################

LR_model = LogisticRegression(
    class_weight="balanced",
    solver="lbfgs",
    penalty="l2",
    random_state = SEED
)


LR_param_grid = {
    "C": [10, 100, 1000, 10000],
}

LR_model = train_final_model(cfg, LR_model, LR_param_grid, "Logistic_Regression")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Logistic_Regression
CV Folds        : 5

Hyperparameter Grid:
C                        : [10, 100, 1000, 10000]
Fitting 5 folds for each of 4 candidates, totalling 20 fits

--------------------------------------------------------------------------------
Total time      : 8.17 seconds
--------------------------------------------------------------------------------
Best params for Logistic_Regression:
{'C': 10}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [10]:
################################
# Decision Tree
################################

DT_model = DecisionTreeClassifier(
    class_weight = "balanced",
    max_depth = 6,
    random_state = SEED
)

DT_param_grid = {
    "max_leaf_nodes": [10, 15, 20, 25, 30],
    "min_samples_leaf": [5, 10, 15],
    "criterion": ["gini", "entropy"],
}
DT_model = train_final_model(cfg, DT_model, DT_param_grid, "Decision_Tree")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Decision_Tree
CV Folds        : 5

Hyperparameter Grid:
max_leaf_nodes           : [10, 15, 20, 25, 30]
min_samples_leaf         : [5, 10, 15]
criterion                : ['gini', 'entropy']
Fitting 5 folds for each of 30 candidates, totalling 150 fits

--------------------------------------------------------------------------------
Total time      : 0.42 seconds
--------------------------------------------------------------------------------
Best params for Decision_Tree:
{'criterion': 'entropy', 'max_leaf_nodes': 15, 'min_samples_leaf': 5}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [11]:
################################
# SVM With RBF Kernel
################################

SVM_model = SVC(
    kernel = "rbf",
    decision_function_shape = 'ovr',
    class_weight = "balanced",
    random_state = SEED,
    probability=True
)

SVM_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.1]
}

SVM_model = train_final_model(cfg, SVM_model, SVM_param_grid, "SVM")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : SVM
CV Folds        : 5

Hyperparameter Grid:
C                        : [0.1, 1, 10, 100]
gamma                    : ['scale', 'auto', 0.01, 0.1]
Fitting 5 folds for each of 16 candidates, totalling 80 fits

--------------------------------------------------------------------------------
Total time      : 1.50 seconds
--------------------------------------------------------------------------------
Best params for SVM:
{'C': 10, 'gamma': 'auto'}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
--------------------------------------------------------------------------------


In [12]:
################################
# Random Forest
################################

RF_model = RandomForestClassifier(
        class_weight = "balanced_subsample",
        random_state = SEED
)


RF_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", 0.3, 0.4],
    "criterion": ['gini', 'entropy'],
    "min_samples_split": [2, 5],
}  

RF_model = train_final_model(cfg, RF_model, RF_param_grid, "Random_Forest")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : Random_Forest
CV Folds        : 5

Hyperparameter Grid:
n_estimators             : [100, 200, 300]
max_depth                : [5, 10, None]
max_features             : ['sqrt', 0.3, 0.4]
criterion                : ['gini', 'entropy']
min_samples_split        : [2, 5]
Fitting 5 folds for each of 108 candidates, totalling 540 fits

--------------------------------------------------------------------------------
Total time      : 30.21 seconds
--------------------------------------------------------------------------------
Best params for Random_Forest:
{'criterion': 'gini', 'max_depth': 10, 'max_features': 0.3, 'min_samples_split': 5, 'n_estimators': 100}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
----------------------------------------------------------------------------

In [13]:
################################
# XGBoost
################################

XG_model = BalancedXGBClassifier(
        random_state = SEED,
        sampling_method = "uniform",
        objective= "multi:softmax",
        eval_metric="mlogloss",
)

XG_param_grid = {
    "n_estimators": [100, 300],
    "max_depth": [5, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "gamma": [0, 0.3],
    "reg_lambda": [1, 5]
}

XG_model = train_final_model(cfg, XG_model, XG_param_grid, "XGBoost")


TRAIN MODEL FOR PRODUCTION ON ALL AVAILABLE DATA
Model Name      : XGBoost
CV Folds        : 5

Hyperparameter Grid:
n_estimators             : [100, 300]
max_depth                : [5, 10]
learning_rate            : [0.01, 0.1]
subsample                : [0.8, 1.0]
colsample_bytree         : [0.8, 1.0]
gamma                    : [0, 0.3]
reg_lambda               : [1, 5]
Fitting 5 folds for each of 128 candidates, totalling 640 fits

--------------------------------------------------------------------------------
Total time      : 22.25 seconds
--------------------------------------------------------------------------------
Best params for XGBoost:
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}

--------------------------------------------------------------------------------
Model and Encoder saved in directory: C:\Users\david\Documents\MASTER_CIENCIA_DE_DATOS\TFM\PROJECT\output\models
-------------

<a id="local"></a>
## 1.2 Interpretabilidad local

Tras concluir el análisis de interpretabilidad global, se llevará a cabo una discusión conjunta de los resultados de rendimiento e interpretabilidad con el objetivo de seleccionar el modelo transparente y el modelo de caja negra que mejore resultados han obtenido. Sobre esta pareja de modelos se realizará el análisis de interpretabilidad local, correspondiente a la segunda fase del estudio.

El objetivo de esta fase consiste en explicar predicciones específicas realizadas por los modelos, es decir, determinar qué variables predictoras conducen a que un complejo residencial concreto sea clasificado dentro de un determinado grado de cerramiento. En particular, este análisis local se centra en el estudio de las siguientes casuísticas:
* **Casos de consenso (acierto mutuo)**. Analizar si, en aquellas instancias correctamente clasificadas por ambos modelos, las explicaciones obtenidas comparten una lógica interpretativa común y coherente con el conocimiento del dominio.
* **Casos de discrepancia**. Analizar cómo el modelo de mayor rendimiento (típicamente el "caja negra") resuelve con éxito aquellas observaciones donde el modelo alternativo comete un error de clasificación. Este análisis resulta especialmente relevante para evaluar si los aciertos en los casos de mayor complejidad se sustentan en relaciones coherentes con el conocimiento del dominio o si carecen de lógica interpretativa.
* **Casos erróneos**. Identificar aquellas instancias en las que ambos modelos presentan errores de clasificación y analizar las posibles causas de dichos errores.

Para realizar este análisis se emplea la técnica de interpretabilidad post-hoc SHAP (SHapley Additive exPlanations) con el objetivo de disponer de un marco de interpretación común para la comparación entre los modelos. SHAP permite cuantificar la contribución individual de cada variable a la predicción realizada para una muestra concreta, proporcionando explicaciones locales comparables independientemente del algoritmo.

Este análisis de interpretabilidad local no se realiza sobre los modelos finales entrenados con el conjunto completo de datos, utilizados en el análisis de interpretabilidad global.

Si se utilizasen estos modelos las predicciones obtenidas para cada muestra no representarían un escenario de inferencia real, ya que dichas muestras habrían sido utilizadas previamente durante el entrenamiento. En consecuencia, tanto las predicciones como las explicaciones asociadas a las mismas estarían sesgadas por el conocimiento previo del algoritmo sobre esas instancias, distorsionando la validez del análisis.

Para aproximar el comportamiento que tendría el modelo en un entorno real de producción ante datos no observados, se implementa una estrategia de predicción y explicabilidad out-of-fold (OOF).

Para ello, se reutilizan las particiones externas (Outer Loop) de la Nested Cross Validation junto con los modelos óptimos obtenidos en cada una de ellas. De este modo, para cada partición se sigue el siguiente procedimiento:
* El modelo configurado con los hiperparámetros óptimos obtenidos durante la Inner Cross Validation se entrena utilizando exclusivamente el conjunto de entrenamiento de la partición.
* Se generan las predicciones out-of-fold sobre el conjunto de test de la partición.
* Finalmente, se calculan los valores SHAP correspondientes a estas predicciones out-of-fold.

Repitiendo este procedimiento sobre todas las particiones externas, se obtienen predicciones y explicaciones SHAP para la totalidad del conjunto de datos, garantizando que cada muestra sea predicha y explicada mediante un modelo que no ha sido entrenado previamente con ella. Esto permite realizar un análisis de interpretabilidad local bajo un escenario metodológicamente representativo del comportamiento esperado en producción.

<a id="mecanismos"></a>
# 2 Mecanismos de interpretabilidad

En esta sección se describen los mecanismos de interpretabilidad empleados durante el análisis.

<a id="coeficientes"></a>
## 2.1 Regresión logística: análisis de coeficientes

El modelo de regresión logística multinomial ofrece un elevado grado de interpretabilidad gracias a la relación directa existente entre las variables predictoras y los coeficientes aprendidos por el modelo. Estos coeficientes describen cómo cada variable influye sobre la probabilidad de pertenencia a las distintas categorías de la variable objetivo. Su interpretación se basa principalmente en dos aspectos: el signo y la magnitud.


El signo del coeficiente indica la dirección del efecto que produce una variable predictora sobre la probabilidad de pertenencia a una determinada clase: 
* $\beta > 0$. Un coeficiente positivo indica que la presencia de la característica incrementa la probabilidad de pertenencia a esa clase frente al resto de categorías. Estas variables pueden interpretarse como factores favorecedores o catalizadores de dicha clase.
* $\beta < 0$. Un coeficiente negativo indica que la presencia de la característica reduce la probabilidad de pertenencia a esa clase. Estas variables pueden interpretarse como factores inhibidores.


La magnitud del coeficiente refleja la intensidad de la contribución de una variable al valor del *log-odds* asociado a cada clase. Dado que la regresión logística multinomial modela relaciones lineales y aditivas en la escala de los *log-odds*, cada coeficiente contribuye sumando o restando, al cálculo de la probabilidad estimada. En consecuencia, coeficientes con magnitudes elevadas, tanto positivas como negativas, indican una mayor influencia de la variable sobre la predicción del modelo, mientras que valores próximos a cero reflejan una capacidad discriminativa baja.

La interpretación de la magnitud de los coeficientes depende de la escala de las variables predictoras. En este conjunto de datos, todas las variables con categórico binarias, lo que permite comparar directamente los coeficientes entre sí. En consecuencia, la interpretación se centra en analizar el efecto que produce la presencia ($1$) o ausencia ($0$) de una determinada característica, manteniendo constantes el resto de variables.

El análisis de interpretabilidad basado en los coeficientes permite abordar dos niveles de estudio complementarios:
* **Análisis global de importancia de variables**. Permite identificar las variables predictoras más relevantes para el modelo en su conjunto. Para ello, se calcula el valor medio del coeficiente en valor absoluto a través de todas las clases, considerando más influyentes aquellas variables con mayores magnitudes promedio.
* **Análisis interpretativo por clase**. Permite caracterizar cada categoría de la variable objetivo a partir de las variables con mayor influencia sobre su probabilidad de pertenencia. En este caso, el signo del coeficiente es relevante para identificar si la variable impacta de forma catalizadora o inhibidora. Este componente interpretativa enriquece el análisis semántico de cada categoría.

<a id="arbol"></a>
## 2.2 Arbol de decisión: reglas de decisión y reducción de impureza

Para la interpretación del árbol de decisión se emplean dos mecanismos intrínsecos al propio modelo que permiten explicar su comportamiento: la importancia de las variables basada en la reducción de impureza y el análisis de las reglas de decisión.

La importancia de una variable en un árbol de decisión se define como la reducción acumulada de impureza que dicha variable produce a lo largo de todos los nodos en los que es utilizada para realizar una partición. En este sentido, una variable es más relevante cuanto mayor es su contribución a la obtención de subconjuntos más homogéneos respecto a la variable objetivo. Este análisis proporciona una visión global del modelo, permitiendo identificar qué información utiliza el árbol para realizar la clasificación.

Para profundizar en el comportamiento del modelo, se analiza su estructura interna mediante la visualización del árbol de decisión, a partir de la cual es posible extraer las reglas de decisión asociadas a las hojas terminales. Cada camino desde el nodo raíz hasta una hoja puede expresarse como una regla de tipo: “si se cumplen las condiciones $x_1, x_2, \dots, x_n$, entonces la clase predicha es $k$”.

Estas reglas permiten identificar las combinaciones de variables que el modelo utiliza para asignar cada clase. La fiabilidad de cada regla está determinada por la distribución de clases en la hoja correspondiente: hojas con una alta concentración de una única clase indican decisiones más seguras, mientras que distribuciones más uniformes reflejan mayor incertidumbre o ambigüedad en la clasificación.

En consecuencia, la interpretabilidad del árbol de decisión depende tanto de la relevancia de las variables identificadas como de la capacidad de las reglas extraídas para ser coherentes con el conocimiento del dominio y suficientemente discriminativas para separar las distintas clases.

<a id="shap"></a>
## 2.3 Modelos de caja negra: SHAP (SHapley Additive exPlanations)

Para poder realizar una aproximación al comportamiento de los modelos SVM, Random Forest y XGBoost se recurre a la técnica de explicabilidad post-hoc SHAP (SHapley Additive exPlanations), introducida por Lundberg y Lee en 2017. Esta técnica ha adquirido una especial relevancia en el contexto actual, donde el uso de modelos cada vez más complejos para alcanzar un alto rendimiento predictivo plantea el desafío de comprender por qué dichos modelos toman determinadas decisiones.

SHAP pertenece a la familia de métodos de atribución aditiva de características, ya que descompone la predicción de un modelo como la suma de las contribuciones individuales de cada variable de entrada. De este modo, permite cuantificar cuánto aporta cada característica a la desviación de la predicción respecto a un valor de referencia, conocido como valor base, que corresponde al valor esperado de la salida del modelo sobre el conjunto de entrenamiento. En otras palabras, el valor base es la predicción que realizaría el modelo sobre un dato del cual no conoce ninguna de sus características.

Bajo este enfoque, la explicación de una predicción se expresa como una combinación lineal de contribuciones asociadas a cada variable, denominadas valores SHAP. Estas contribuciones se fundamentan en la teoría de juegos cooperativos, específicamente en los valores de Shapley propuestos por Lloyd Shapley en 1953. Dicha teoría distribuye de forma equitativa la ganancia total de un juego entre los distintos participantes en función de su contribución marginal al resultado final. En el contexto del aprendizaje automático, el “juego” corresponde a la predicción de una instancia concreta, mientras que los “jugadores” son las variables de entrada del modelo.


SHAP se ha consolidado como uno de los marcos de interpretabilidad más utilizados debido a su capacidad para unificar diversos métodos previos bajo un mismo fundamento teórico. Los autores muestran que técnicas como LIME, DeepLIFT, Layer-Wise Relevance Propagation (LRP), Shapley regression values, Shapley sampling values o Quantitative Input Influence pueden interpretarse como aproximaciones a los valores de Shapley, diferenciándose únicamente en la estrategia de estimación empleada.

Además, SHAP proporciona un marco axiomático sólido, ya que es el único método de atribución aditiva que satisface simultáneamente tres propiedades fundamentales:

* Exactitud local (local accuracy): la suma de las contribuciones de todas las características reproduce exactamente la predicción del modelo para cada observación.
* Ausencia (missingness): las características ausentes no contribuyen a la explicación.
* Consistencia (consistency): si un cambio en el modelo incrementa la contribución de una característica, su valor SHAP no puede disminuir.

Estas propiedades garantizan matemáticamente una solución única para los valores SHAP, algo de lo que carecen los métodos previos. Los autores de SHAP argumentan, que estas propiedades se alinean con la intuición humana y que su ausencia puede dar lugar a explicaciones percibidas como inconsistentes o poco intuitivas. Para respaldar esta afirmación, presentan los resultados empíricos de experimento en los que las explicaciones de SHAP muestran mayor concordancia con evaluaciones humanas en comparación con otros métodos como LIME o DeepLIFT.

Por otro lado, SHAP proporciona interpretabilidad tanto a nivel local como global, a diferencia de otros métodos que se limitan exclusivamente a explicaciones globales. Los valores SHAP se calculan a nivel de instancia, lo que permite descomponer de forma directa la predicción de cada observación en función de la contribución de sus características más influyentes. Para extender esta interpretabilidad local a una visión global del modelo, los valores SHAP se agregan a lo largo de todo el conjunto de datos, obteniéndose así una medida de importancia de las variables basada en la magnitud media de sus contribuciones locales.

<a id="shap_teoria"></a>
### 2.3.1 Fundamento teórico

Sea un modelo modelo $f(x)$ que se desea explicar para una observación específica $x$, SHAP establece una función de explicación $g(z')$ a través de un modelo lineal tal que:

$$g(z') = \phi_0 + \sum_{i=1}^{M} \phi_i z'_i$$

donde:
* $z' \in \{0, 1\}^M$ es un vector de presencia de características en el que $z'_i = 1$ significa que la característica $i$ está observada, y $0$ que está ausente.
* $M$ es el número total de características (variables independientes)
* $\phi_i \in \mathbb{R}$ es el valor SHAP de la característica $i$.
* $\phi_0$ es el valor base o predicción esperada.

El valor SHAP ($\phi_i$) para la característica $i$ se define mediante la ecuación clásica de *Shapley*:
$$\phi_i(f, x) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(M - |S| - 1)!}{M!} \left[ f_x(S \cup \{i\}) - f_x(S) \right]$$

donde:
* $F$ es el conjunto de todas las características.
* $S$ es un subconjunto de características que no incluye a la característica $i$.
* $f_x(S)$ es la función que predice la salida utilizando solo las características presentes en el subconjunto $S$. Esta función se denomina valor de la coalición.
* $\left[ f_x(S \cup \{i\}) - f_x(S) \right]$ es la contribución marginal de la característica $i$ al subconjunto $S$.
* $\frac{|S|!(M - |S| - 1)!}{M!}$ es el factor de ponderación, que representa la probabilidad de que el subconjunto $S$ aparezca en una permutación aleatoria de las características.

El valor SHAP es por tanto el valor promedio ponderado de la contribución marginal de la característica $i$ a través de todas las combinaciones de variables posibles.

Los modelos de aprendizaje automático no están diseñados para recibir subconjuntos de variables como entrada. Por lo tanto, para evaluar $f_x(S)$, cuando faltan características, se define el concepto de predicción esperada. Formalmente, el valor de una coalición $S$ se define como la esperanza condicional de la predicción del modelo, dado que conocemos los valores de las características en $S$:

$$f_x(S) = \mathbb{E}_{X_{\setminus S} | X_S = x_S} [f(X)]$$

Donde $X_S = x_S$ son los valores observados de la instancia actual para las características del grupo $S$, y $X_{\setminus S}$ representa las características ausentes. Cuando el subconjunto $S$ es el conjunto vacío ($\emptyset$), significa que no conocemos el valor de ninguna característica de nuestra instancia $x$. En este caso:
$$\phi_0 = f_x(\emptyset) = \mathbb{E}[f(X)]$$

Por lo tanto, la predicción esperada ($\phi_0$) se define formalmente como la esperanza matemática de las predicciones del modelo sobre toda la distribución de los datos de entrenamiento. Es la predicción que el modelo haría "a ciegas" si no conociese absolutamente nada sobre la observación actual.

En la práctica calcular la esperanza condicional exacta $\mathbb{E}[f(X) | X_S = x_S]$ es computacionalmente intratable puesto que implica conocer la distribución conjunta de los datos. SHAP resuelve esto mediante dos aproximaciones dependiendo del algoritmo:
* Aproximación de Independencia (KernelSHAP). Asume que las características presentes ($S$) y ausentes ($\setminus S$) son independientes. Bajo esta suposición, la esperanza condicional se convierte en una esperanza marginal. En la práctica esto se implementa utilizando una muestra de fondo (un background dataset) y reemplazando las características ausentes con los valores de esos datos de fondo. Esta ha sido la técnica empleada para calcular los valores SHAP del modelo SVM.
* TreeSHAP para modelos basados en árboles. Para algoritmos como XGBoost y Random Forest, TreeSHAP calcula $\mathbb{E}[f(X) | X_S = x_S]$ de manera exacta y eficiente en tiempo polinomial. Lo logra siguiendo los caminos del árbol de decisión: si una división (split) utiliza una variable ausente (que no está en $S$), el algoritmo sigue ambos caminos del split simultáneamente y pondera las predicciones de las hojas resultantes por la proporción de registros de entrenamiento que fluyen a través de cada rama (citar *Lundberg, S.M., Erion, G., Chen, H. et al. From local explanations to global understanding with explainable AI for trees. Nat Mach Intell*).

<a id="shap_interp"></a>
### 2.3.2 Interpretabilidad de los valores SHAP

Para la implementación de SHAP se ha utilizado la librería de Python shap, desarrollada por los propios autores del método. En el caso de problemas de clasificación multiclase, los valores SHAP se calculan para cada característica y para cada clase de la variable objetivo, de forma que cada predicción queda descompuesta en contribuciones específicas por clase.

De manera análoga a los coeficientes de la regresión logística, los valores SHAP se interpretan a partir de su signo y su magnitud. Sin embargo, mientras que en la regresión logística los coeficientes son parámetros globales del modelo, es decir, existe un único valor por característica y clase que describe su efecto general, en SHAP la explicación se construye a nivel de instancia. Esto implica que, para cada muestra individual, se obtiene un conjunto de valores SHAP que cuantifica la contribución de cada variable a la predicción de cada clase en ese caso concreto.

Sin embargo, en SHAP los valores SHAP se obtienen a nivel de instancia, de modo que para cada muestra se tienen los valores SHAP de todas sus caracteristicas, para el valor correspondiente que tengan, respecto a cada categoría de la variable objetivo. La interpretación local de estos valores hace referencia a la muestra individual.

El signo del valor SHAP indica la dirección del efecto de una variable sobre la salida del modelo para una clase específica en una observación determinada:
$\phi > 0$. Indica que la variable contribuye a incrementar la salida asociada a dicha clase respecto al valor base del modelo, actuando como un factor que favorece esa predicción.
$\phi < 0$. Indica que la variable contribuye a disminuir la salida asociada a dicha clase respecto al valor base, actuando como un factor inhibidor.

Por su parte, la magnitud del valor SHAP refleja la intensidad de dicha contribución, es decir, el grado en que una variable desplaza la predicción respecto al valor base.

A partir de la agregación de los valores SHAP a lo largo de todas las instancias del conjunto de datos es posible obtener una medida de importancia global de las variables. Asimismo, el análisis de la distribución de los valores SHAP en función del valor que toma cada característica (en este caso, 0 o 1 al tratarse de variables binarias) permite estudiar el comportamiento global de cada variable, tanto en términos de su influencia media como de la dirección de su efecto sobre las distintas clases.